# Comparison of Python and REST API calls

In [ ]:
# setup
import sys
if ".." not in sys.path:
    sys.path.append("..")
import requests
from IPython.display import JSON
API_URL = "http://genra_api:5000/genra-api/api/genra/"

## Health check

No Python API

In [ ]:
JSON(requests.get(API_URL + "v3/healthCheck/").json())

## FP IDs

Depends on deployment type

In [ ]:
from genraweb.lib.fp.fpclass import FPGen
FPGen.FPClass

## Search

In [ ]:
from genraweb.resources import DB
JSON(next(DB.compounds.find({"name": "Bisphenol A"}, {"_id": 0})))

But better to use `search_chems` which searches synonyms etc. etc.

In [ ]:
from genraweb.routes.searchChem_grouped import search_chems
JSON(search_chems("BPA"))  # RDKit complains about BPA not being SMILES, ignore it

...and the corresponding REST call...

In [ ]:
JSON(requests.get(API_URL + "v3/searchChems/?txt=BPA").json())

## Setup

No Python API

In [ ]:
JSON(requests.get(API_URL + "v4/uiSetup?chem_id=DTXCID30182").json())

## Radial view

Note results are not quite the same

In [ ]:
from genraweb.lib.mongofp_NN import searchFP
nghbrs = searchFP("DTXCID30182", fp="chm_mrgn", sel_by="tox_txrf", s0=0.1, max_hits=10 + 1)
JSON(nghbrs)

In [ ]:
JSON(requests.get(
    API_URL + "v4/uiRadialView?chem_id=DTXCID30182&fp=chm_mrgn&sel_by=tox_txrf&s0=0.1&k0=10"
).json())

## FP heat map

Now Python and REST API results very different, REST result AG Grid flavored

In [ ]:
from genraweb.lib.fp.fputils import fp_counts_for_chems
chem_ids = [i["chem_id"] for i in nghbrs]
JSON(fp_counts_for_chems(chem_ids=chem_ids))

In [ ]:
JSON(requests.get(
    API_URL + "v4/uiFingerPrintHeatChart?chem_id=DTXCID30182&fp=chm_mrgn&sel_by=tox_txrf&s0=0.1&k0=10"
).json())

## Assay list (panel 3)

In [ ]:
from genraweb.lib.fp.fputils import get_toxref_assays_for_chems, readacross_table
# readacross_table is ~ the uiAssayList REST result
# readacross_table(target_chem_id="DTXCID30182", fp_id="chm_mrgn", sel_by="tox_txrf", s0=0.1, k0=10 + 1))
get_toxref_assays_for_chems(chem_ids=chem_ids)  # Pandas DF

In [ ]:
JSON(requests.get(
    API_URL + "v4/uiAssayList?chem_id=DTXCID30182&fp=chm_mrgn&sel_by=tox_txrf&s0=0.1&k0=10"
).json())

## Generate Read Across

Python API wise this would just be `get_toxref_assays_for_chems()` again as above, apart from PhysProp data

In [ ]:
from genraweb.lib.properties.physprop import ID2PP, chem_props
ID2PP

In [ ]:
JSON(chem_props(chem_ids))

In [ ]:
gra_data = requests.get(
    API_URL + "v4/uiGenerateReadAcross?chem_id=DTXCID30182&fp=chm_mrgn&sel_by=tox_txrf&s0=0.1&k0=10"
).json()
JSON(gra_data)

## Run read-across

In [ ]:
from genraweb.lib.genrapred import runGenRA
predictions = runGenRA(
    "DTXCID30182",
    CID=chem_ids,
    DB=DB,
    fp_x="chm_mrgn",
    fp_y="toxp_txrf",
    sel_by="tox_txrf",
    metric="jaccard",
    k0=10,
    s0=0.1,
    pred=True,
    ret="df",
    n_perm=200,
    pos_min=1,
    neg_min=1,
)
JSON(predictions)

In [ ]:
post_data = {
    "fp": "chm_mrgn",
    "k0": 10,
    "s0": 0.1,
    "dsstox_cid": "DTXCID30182",
    "sel_by": "tox_txrf",
    "neg0": 1,
    "pos0": 1,
    "chem_inc": [
        {"isChecked": True, "chem_id": i} for i in chem_ids
    ],
    "tox_inc": [],  # which assays,  [] => all
}
rra_data = requests.post(
    API_URL + "v4/uiRunReadAcross", json=post_data
).json()
JSON(rra_data)

## Download

In [ ]:
post_data = {
    "fp": "chm_mrgn",
    "k0": 10,
    "s0": 0.1,
    "chem_id": "DTXCID30182",
    "sel_by": "tox_txrf",
    "neg0": 1,
    "pos0": 1,
    "chem_inc": [
        {"isChecked": True, "chem_id": i} for i in chem_ids
    ],
    "tox_inc": [],  # which assays,  [] => all
    "rra": True,  # False for "Generate Read Across version"
}
download = requests.post(
    API_URL + "v4/uiDownload/xlsx", json=post_data
)
from io import BytesIO
dl = BytesIO(download.content)
dl.seek(0)
import pandas as pd
pd.read_excel(dl, sheet_name="Metadata")